# RSNA Knee Abnormality Detection: understanding the data before modelling it

This notebook covers three things in order: what the competition is asking for, the medical
and imaging concepts needed to interpret the data correctly, and an exploratory analysis of
the actual files. The goal is to leave with a clear picture of what a study contains, what the
labels do and do not cover, and which parts of the dataset are easy to misuse without any error
being raised in the process. A companion baseline notebook builds a model on top of what is
established here.

## The task

The RSNA Knee Abnormality Detection challenge asks participants to detect twelve clinically
relevant findings from knee MRI examinations. Each exam, referred to as a study, is scored
independently for each of the twelve findings, and the competition metric is the
macro-averaged ROC-AUC across them, meaning all twelve findings are weighted equally regardless
of how common they are.

A few practical points about the competition, taken from the official challenge announcement
rather than assumed:

* It is a code competition: submissions are notebooks that run against a hidden test set with
  internet access disabled during scoring. Any pretrained weights used at inference time need
  to be attached as a Kaggle dataset ahead of time, since nothing can be downloaded at
  submission time.
* The dataset contains more than 5,000 knee MRI exams from sites spread across several
  continents, each with an accompanying radiology report written in one of roughly a dozen
  languages. This is the first RSNA challenge to combine imaging with free-text multilingual
  reports rather than structured labels alone.
* The evaluation set was annotated by expert musculoskeletal radiologists, but, as the
  exploration further down shows, only a small fraction of studies carry that direct
  image-derived annotation; the rest arrive with a report and nothing else.

## The twelve findings

The labels fall into four groups anatomically. Working through them matters here because
several only make sense once "medial" and "lateral" are pinned to a physical side of the knee,
which turns out to be one of the trickier parts of this dataset.

Ligaments:
* ACL (anterior cruciate ligament) tear: the ACL runs diagonally through the center of the
  knee and stabilizes forward sliding of the shin bone relative to the thigh bone. It is one of
  the most commonly injured structures in sports medicine.
* MCL (medial collateral ligament) tear: a band running along the inner, medial side of the
  knee that resists sideways force which would otherwise open the joint outward.

Menisci, two crescent-shaped cartilage pads that cushion the joint and distribute load between
the thigh and shin bones:
* Medial meniscus tear: the pad on the inner side of the knee, injured more frequently than
  its counterpart.
* Lateral meniscus tear: the pad on the outer side.

Osteoarthritis (OA), the degenerative loss of the cartilage lining the joint surfaces, graded
separately for the three compartments of the knee:
* Medial compartment OA: between the inner thigh bone and inner shin bone.
* Lateral compartment OA: between the outer thigh bone and outer shin bone.
* Patellofemoral (PF) OA: between the kneecap and the groove of the thigh bone it slides in.

Soft tissue and other findings:
* Effusion: excess fluid inside the joint capsule, a nonspecific sign of irritation or injury.
* Synovitis: inflammation of the synovium, the membrane lining the joint.
* Baker's cyst: a fluid-filled swelling behind the knee, formed when joint fluid is pushed into
  a bursa there.
* Contusion: a bone bruise, a marrow-level injury that occurs without a visible fracture line,
  usually from impact or an associated ligament injury.
* Fracture: a break in one of the bones forming the joint.

Five of these twelve labels, the medial and lateral meniscus, medial and lateral OA, and the
MCL, depend on knowing which knee, left or right, is being examined, since medial and lateral
are defined relative to the body's midline and therefore land on opposite sides of the image
depending on laterality. That dependency resurfaces later, in the DICOM header section.

## MRI concepts needed to read this data

An MRI study is not one image, or even one 3D volume; it is a collection of separately
acquired series, and a knee study typically contains several. Two properties of a series
matter here in particular.

Imaging plane. Series are acquired in one of three standard planes: sagittal, a side-on slice
cutting the knee front to back; coronal, a front-facing slice cutting top to bottom, the way an
ordinary photograph of a knee would look; and axial, a cross-sectional slice cutting the knee
horizontally into top and bottom halves. Different findings tend to show up more clearly in
different planes. Ligament tears are usually best seen on sagittal images, while cartilage and
meniscus damage often benefit from a coronal or axial view as well.

Contrast weighting and fat suppression. The same tissue can look completely different depending
on how the scanner is tuned. T1-weighted images show anatomy with good structural detail but
poor fluid contrast. T2-weighted and proton-density images make fluid appear bright, which is
what makes effusions, edema and many tears visible. Fat suppression is a separate technique,
applied on top of any of the above, that suppresses the signal from fat so that adjacent fluid
or edema is not masked by a bright fat signal. In principle a sequence can be fluid-sensitive
and fat-suppressed, fluid-sensitive without fat suppression, or a T1 sequence with or without
fat suppression: two independent design choices. Whether the dataset actually keeps them
independent is checked directly in the exploration below.

Because a full knee protocol usually captures the same anatomy in more than one plane and more
than one weighting, a study is best thought of as a small, irregularly sized collection of
series rather than a single fixed-size volume. That irregularity, and how to handle it, is the
central engineering problem in this competition.

## How this notebook is organized

1. Loading the provided files and confirming what each one contains.
2. How much of the data carries the twelve labels, and what the rest looks like.
3. Label prevalence and co-occurrence on the labelled subset.
4. What a study is made of: series counts, planes, and acquisition types.
5. The free-text reports: language, length, and duplicate reports.
6. What the DICOM headers add beyond the two CSV files, including laterality and slice order.
7. A visual look at an example study.
8. A summary of the practical consequences for modelling, carried forward into the baseline.

Every number quoted in the markdown below is produced by the code cell that follows it, so the
figures can be checked directly by running the notebook against the competition data.

## 1. Loading the data

The cell below sets a consistent plotting style and locates the competition files. Kaggle
mounts a competition's data at a slightly different path depending on whether it was attached
through the notebook editor or pushed through the Kaggle API, so the search below walks
`/kaggle/input` looking for the expected file layout rather than assuming a fixed path. This
avoids a notebook that works interactively but fails the moment it is submitted through a
different attachment path.

In [ ]:
import os
import re
import unicodedata
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings("ignore")

# A small, deliberately restrained plotting style: one colour per imaging plane, used
# consistently in every figure, and a single sequential blue scale for magnitudes.
PLANE_COLORS = {"Sagittal": "#2a6fdb", "Coronal": "#e1732c", "Axial": "#1f9e77"}
BLUE, GREY, RED = "#2a6fdb", "#8a8a86", "#d94f4f"

mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#d8d8d5", "grid.color": "#ececea",
    "font.size": 10.5, "figure.dpi": 110,
})

def find_competition_root(base="/kaggle/input"):
    # Handles both the short path Kaggle uses when a competition is attached through the
    # notebook editor and the longer /kaggle/input/competitions/<slug>/ path used when a
    # notebook is pushed through the Kaggle API.
    for root, dirs, files in os.walk(base):
        if "train.csv" in files and ("train_series.csv" in files or "train_series" in dirs):
            return root
        # Do not descend into the (large) per-study image directories while searching.
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
    raise FileNotFoundError(
        "Could not locate the competition files under /kaggle/input. "
        "Attach the RSNA Knee Abnormality Detection dataset to this notebook."
    )

ROOT = find_competition_root()
train = pd.read_csv(f"{ROOT}/train.csv")
test = pd.read_csv(f"{ROOT}/test.csv")
train_series = pd.read_csv(f"{ROOT}/train_series.csv")
test_series = pd.read_csv(f"{ROOT}/test_series.csv")

# The twelve finding columns are every train.csv column apart from the study identifier
# and the free-text report.
TARGETS = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]

print(f"competition root : {ROOT}")
print(f"train.csv        : {train.shape}")
print(f"test.csv         : {test.shape}")
print(f"train_series.csv : {train_series.shape}")
print(f"test_series.csv  : {test_series.shape}")
print(f"\ntrain.csv columns : {list(train.columns)}")
print(f"test.csv columns  : {list(test.columns)}")
print(f"\ntarget columns ({len(TARGETS)}): {TARGETS}")

The comparison between `train.csv` and `test.csv` columns is worth pausing on. If the
`Report` column is present in `train.csv` but absent from `test.csv`, the free-text report can
only ever be used as a training signal, or as an input to a labelling process that runs before
the competition's test set is seen; it is never available as a model input at inference time.
Every study, train or test, does come with an entry in the corresponding `_series.csv` file
describing its imaging series, which is the only input guaranteed to exist at inference.

## 2. How much of the data actually carries the twelve labels

This is the fact that shapes most other modelling decisions, so it comes first. A study in
`train.csv` can be in one of two states: it has a value in every one of the twelve target
columns, meaning it went through direct radiologist annotation, or those columns are empty and
only the `Report` column carries information. The next cell measures the split.

In [ ]:
has_full_labels = train[TARGETS].notna().all(axis=1)
gold = train[has_full_labels].copy()
n_gold = len(gold)
n_report_only = int((~has_full_labels).sum())

fig, ax = plt.subplots(figsize=(9, 1.6))
ax.barh([0], [n_report_only], color="#e4e9f0", edgecolor="white", linewidth=2, height=0.55)
ax.barh([0], [n_gold], left=[n_report_only], color=BLUE, edgecolor="white", linewidth=2, height=0.55)
ax.text(n_report_only / 2, 0, f"{n_report_only:,} studies, report only", ha="center", va="center", fontsize=10.5)
ax.annotate(f"{n_gold} studies with full labels", xy=(n_report_only + n_gold / 2, 0.28),
            xytext=(max(n_report_only - 500, 0), 0.9), fontsize=10.5, fontweight="bold", color=BLUE,
            arrowprops=dict(arrowstyle="-", color=BLUE))
ax.set_xlim(0, len(train)); ax.set_ylim(-0.6, 1.2); ax.axis("off")
ax.set_title("Where the twelve labels are present in train.csv", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"studies with all twelve labels present : {n_gold} / {len(train):,}  ({100 * n_gold / len(train):.2f}%)")
print(f"studies with a non-empty report        : {(train.Report.fillna('').str.strip() != '').sum():,} / {len(train):,}")
print(f"labelled studies that also have a report : {(gold.Report.fillna('').str.strip() != '').sum()} / {n_gold}")

This means the great majority of supervision available for training is indirect: a
report describing what a radiologist saw, not a set of twelve numbers. Whatever labels get used
for the bulk of training either come from parsing those reports, or from a model trained on the
small labelled subset and then applied to generate targets for the rest. Both routes carry
their own error, and it is worth keeping in mind that any validation metric computed against
report-derived targets is really measuring agreement with a text-parsing process, not
necessarily with the ground truth radiological finding.

The small set of directly labelled studies is the only place where validation and the true
labels coincide. Because it is small, any AUC computed purely on this subset should be treated
as a rough, high-variance estimate: with on the order of a few dozen positive examples for a
label, a handful of prediction swaps can move the AUC noticeably.

## 3. Label prevalence and co-occurrence

Restricting to the directly labelled studies, the next two cells look at how common each
finding is and which findings tend to appear together. Because the competition metric averages
AUC equally across all twelve labels, a rare finding is exactly as important to get right as a
common one, even though it offers far fewer positive examples to learn from.

In [ ]:
prevalence = gold[TARGETS].mean().sort_values()
positive_counts = gold[TARGETS].sum().astype(int)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(prevalence)), prevalence.values * 100, color=BLUE, edgecolor="white", linewidth=2, height=0.68)
for i, (name, value) in enumerate(prevalence.items()):
    ax.text(value * 100 + 1.2, i, f"{value * 100:.1f}%  ({positive_counts[name]}/{n_gold})",
            va="center", fontsize=9.5, color="#444")
ax.set_yticks(range(len(prevalence)), prevalence.index)
ax.set_xlim(0, 85)
ax.set_xlabel("share of the labelled studies positive for this finding")
ax.set_title("Prevalence of each finding among the directly labelled studies", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

n_findings_per_study = gold[TARGETS].sum(axis=1)
print(f"findings per labelled study : mean {n_findings_per_study.mean():.2f}, "
      f"median {n_findings_per_study.median():.0f}, "
      f"range {int(n_findings_per_study.min())}-{int(n_findings_per_study.max())}")
print(f"labelled studies with no positive finding at all : {(n_findings_per_study == 0).sum()}")

In [ ]:
# Jaccard overlap between every pair of findings, computed only on the labelled studies.
M = gold[TARGETS].astype(int)
intersection = M.T.dot(M).astype(float)
jaccard = pd.DataFrame(index=TARGETS, columns=TARGETS, dtype=float)
for a in TARGETS:
    for b in TARGETS:
        union = int(((M[a] == 1) | (M[b] == 1)).sum())
        jaccard.loc[a, b] = intersection.loc[a, b] / union if union else 0.0

off_diagonal = jaccard.values.copy()
np.fill_diagonal(off_diagonal, np.nan)
vmax = float(np.nanmax(off_diagonal))

fig, ax = plt.subplots(figsize=(7.5, 6.3))
cmap = mpl.colormaps["Blues"].copy()
cmap.set_bad("white")
im = ax.imshow(off_diagonal, cmap=cmap, vmin=0, vmax=vmax)
ax.set_xticks(range(len(TARGETS)), TARGETS, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(TARGETS)), TARGETS, fontsize=9)
for i in range(len(TARGETS)):
    for j in range(len(TARGETS)):
        v = jaccard.values[i, j]
        if i != j and v >= 0.25:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if v > vmax * 0.6 else "#222")
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03, label="Jaccard overlap")
ax.set_title("Which findings tend to co-occur (diagonal masked)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

pairs = [(TARGETS[i], TARGETS[j], int(intersection.values[i, j]))
         for i in range(len(TARGETS)) for j in range(i + 1, len(TARGETS))]
print("most frequent co-occurring pairs, count out of", n_gold, "labelled studies:")
for a, b, count in sorted(pairs, key=lambda t: -t[2])[:8]:
    print(f"  {a:18} + {b:18} {count}")

Findings that share an anatomical cause, such as the two osteoarthritis compartments,
or effusion and synovitis, tend to co-occur more than unrelated findings, which is unsurprising
but useful to confirm: a model that shares representations across related labels, rather than
treating all twelve as fully independent, has a reasonable prior to lean on.

## 4. What a study is actually made of

A study is a variable-sized collection of series rather than a fixed volume. This section
measures exactly how variable, using the `_series.csv` metadata rather than opening any image
files yet.

In [ ]:
series_per_study = train_series.groupby("StudyInstanceUID").size()
plane_counts = train_series["Anatomical_Plane"].value_counts()
plane_order = ["Sagittal", "Coronal", "Axial"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6), gridspec_kw={"width_ratios": [1.1, 1]})

ax = axes[0]
counts = series_per_study.value_counts().sort_index()
ax.bar(counts.index, counts.values, color=BLUE, edgecolor="white", linewidth=2, width=0.7)
ax.set_xlabel("series in the study"); ax.set_ylabel("number of studies")
ax.set_title(f"Series per study (mean {series_per_study.mean():.2f}, "
             f"median {series_per_study.median():.0f})", loc="left", fontweight="bold")

ax = axes[1]
values = [int(plane_counts.get(p, 0)) for p in plane_order]
colors = [PLANE_COLORS[p] for p in plane_order]
bars = ax.barh(plane_order[::-1], values[::-1], color=colors[::-1], edgecolor="white", linewidth=2, height=0.6)
for bar, value in zip(bars, values[::-1]):
    ax.text(value * 0.98, bar.get_y() + bar.get_height() / 2, f"{value:,}",
            ha="right", va="center", color="white", fontsize=10, fontweight="bold")
ax.set_xlabel("series"); ax.set_title(f"Series by plane ({len(train_series):,} total)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Crossing plane with the fat-suppression flag gives six candidate acquisition "slots".
# This measures how often each of the six is actually present, and how many studies have
# all six, which matters directly for any model design that expects fixed inputs.
train_series["combo"] = train_series["Anatomical_Plane"] + " / fs=" + train_series["Fat_Suppression"].astype(str)
presence = (train_series.pivot_table(index="StudyInstanceUID", columns="combo",
                                      values="SeriesInstanceUID", aggfunc="count")
            .notna())
combos = [f"{p} / fs={fs}" for p in plane_order for fs in (1, 0)]
presence = presence.reindex(columns=combos, fill_value=False)
share = presence.mean() * 100

fig, ax = plt.subplots(figsize=(9.5, 3.6))
colors = [PLANE_COLORS[c.split(" / ")[0]] for c in combos]
ax.barh(range(len(combos))[::-1], share[combos].values, color=colors, edgecolor="white", linewidth=2, height=0.65)
for i, c in enumerate(combos):
    v = share[c]
    ax.text(v + 1.5, len(combos) - 1 - i, f"{v:.1f}%  ({int(presence[c].sum()):,} studies)",
            va="center", fontsize=9.5, color="#444")
ax.set_yticks(range(len(combos))[::-1], combos)
ax.set_xlim(0, 125)
ax.set_xlabel("share of studies containing at least one such series")
ax.set_title("How often each plane / fat-suppression combination is present", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

n_all_six = int(presence.all(axis=1).sum())
print(f"studies containing all six plane/flag combinations : {n_all_six:,} ({100 * n_all_six / len(presence):.1f}%)")
print("\nnumber of the six combinations present, by study count:")
print(presence.sum(axis=1).value_counts().sort_index().to_string())

If one particular combination, such as a fat-suppressed axial series, turns out to be
present in essentially every study while others are present in only a minority, that has a
direct design consequence: a model that expects six fixed input slots will be imputing most of
them for most studies. A presence mask that explicitly marks which slots were actually
acquired, rather than silently filling missing ones with zeros, lets a model distinguish "this
was never scanned" from "this was scanned and looked unremarkable," which are very different
pieces of information for a network to receive as identical black inputs.

It is also worth checking whether the two flags used to build these combinations, fluid
sensitivity and fat suppression, are actually independent, since the acquisition physics does
allow them to vary independently.

In [ ]:
crosstab = pd.crosstab(train_series["Fluid_Sensitive"], train_series["Fat_Suppression"])
print(crosstab.to_string())
identical = (train_series["Fluid_Sensitive"] == train_series["Fat_Suppression"]).mean() * 100
print(f"\nrows where the two flags agree : {identical:.1f}%")

If that agreement comes out at, or very close to, 100%, the two columns are carrying one
axis of information between them rather than two independent ones in this dataset, even though
fluid sensitivity (a property of the pulse sequence's contrast weighting) and fat suppression
(a preparation applied on top of it) are physically separable. A model design that treats
"plane x weighting x fat-suppression" as three independent input axes using these two columns
would then secretly have only two independent axes, not three. Recovering weighting and fat
suppression separately, where that distinction matters, generally requires reading the DICOM
headers directly, which section 6 returns to.

## 5. The free-text reports

For the great majority of the corpus, the report is the only supervision signal available, so
understanding its properties is as important as understanding the images. Two properties
matter most for building any kind of weak-label pipeline out of the reports: what languages
they are written in, and whether the same report text repeats across different studies.

In [ ]:
reports = train["Report"].fillna("")

def detect_script(text):
    # A coarse, first-alphabetic-character script detector. This is a floor, not a language
    # count: Latin script alone covers many languages, so the true number of languages present
    # is higher than the number of scripts detected here. The point is only to show that a
    # single-language keyword approach will silently return nothing on a meaningful share of
    # the corpus.
    for ch in text:
        if ch.isalpha():
            name = unicodedata.name(ch, "")
            if "CYRILLIC" in name:
                return "Cyrillic"
            if "GREEK" in name:
                return "Greek"
            return "Latin"
    return "empty"

scripts = reports.map(detect_script).value_counts()
lengths = reports.str.len()

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.5), gridspec_kw={"width_ratios": [1, 1.3]})

ax = axes[0]
order = [s for s in ["Latin", "Greek", "Cyrillic", "empty"] if s in scripts.index]
values = [int(scripts[s]) for s in order]
ax.barh(order[::-1], values[::-1], color=BLUE, edgecolor="white", linewidth=2, height=0.6)
for i, v in enumerate(values[::-1]):
    ax.text(v + len(train) * 0.01, i, f"{v:,} ({100*v/len(train):.1f}%)", va="center", fontsize=9.5)
ax.set_title("Report script (a lower bound on language count)", loc="left", fontweight="bold")

ax = axes[1]
ax.hist(lengths, bins=60, color=BLUE, edgecolor="white", linewidth=0.5)
median_len = lengths.median()
ax.axvline(median_len, color=RED, linestyle="--", linewidth=1.3)
ax.text(median_len, ax.get_ylim()[1] * 0.92, f"  median {median_len:.0f}", color=RED, fontsize=9)
ax.set_xlabel("characters"); ax.set_ylabel("studies")
ax.set_title(f"Report length (max {lengths.max():,} characters)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

A keyword search written for a single language will return nothing on every report that
is not written in that language, and a missing match looks identical to a genuine negative
finding unless the pipeline explicitly tracks language or script coverage. Any weak-labelling
approach based on report text needs to either support multiple languages or clearly flag the
studies it could not confidently parse, rather than defaulting them to negative.

The second property, duplicate reports, matters for validation rather than for parsing.

In [ ]:
report_counts = reports.value_counts()
non_empty = report_counts[report_counts.index != ""]
duplicated_texts = non_empty[non_empty > 1]

print(f"unique non-empty report texts       : {len(non_empty):,}")
print(f"report texts shared by more than one study : {len(duplicated_texts):,}")
print(f"studies sitting inside such a duplicate group : {int(duplicated_texts.sum()):,}")
if len(duplicated_texts):
    print(f"largest single duplicate group : {int(duplicated_texts.iloc[0]):,} studies")

group_sizes = duplicated_texts.value_counts().sort_index()
if len(group_sizes):
    fig, ax = plt.subplots(figsize=(8.5, 3))
    ax.bar(group_sizes.index.astype(str), group_sizes.values, color=RED, edgecolor="white", linewidth=2, width=0.7)
    ax.set_xlabel("studies sharing one identical report text")
    ax.set_ylabel("number of such groups")
    ax.set_title("Duplicate report groups", loc="left", fontweight="bold")
    plt.tight_layout(); plt.show()

A duplicated report is most plausibly a template reused for an unremarkable exam. Any
label derived from report text will then be identical for every study inside that group. If a
random train/validation split happens to place some members of a duplicate group in training
and others in validation, the validation score for those studies partly reflects having seen an
identical target during training rather than genuine generalisation. Grouping the
cross-validation split by a hash of the report text, so that studies sharing a report always
land on the same side of a split, removes this specific leak. This is carried into the fold
construction in the baseline notebook.

## 6. What the DICOM headers add

The two CSV files describe series at a coarse level: which study, which series, which plane,
which flags. They do not say which knee, left or right, was scanned, and they do not describe
the order in which the individual image files within a series should be read. Both of those
have to come from the DICOM headers of the underlying image files.

This section reads one header per series for a random sample of studies. Reading only the
header, without decoding the pixel data, keeps this fast enough to run over a reasonably large
sample.

In [ ]:
import pydicom
from concurrent.futures import ThreadPoolExecutor

HEADER_TAGS = [
    "Laterality", "ImageLaterality", "PixelSpacing", "Rows", "Columns",
    "RepetitionTime", "EchoTime", "ScanningSequence", "ScanOptions",
    "SeriesDescription", "Manufacturer", "ManufacturerModelName",
    "ImagePositionPatient", "ImageOrientationPatient", "SliceThickness", "InstanceNumber",
]
N_SAMPLE_STUDIES = 400

all_study_ids = sorted(os.listdir(f"{ROOT}/train_series"))
rng = np.random.default_rng(0)
sample_ids = rng.choice(all_study_ids, size=min(N_SAMPLE_STUDIES, len(all_study_ids)), replace=False)

def read_one_series_header(study_id, series_id, study_dir):
    series_dir = f"{study_dir}/{series_id}"
    files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
    if not files:
        return None
    # The middle file is representative enough for header-level tags such as spacing or
    # laterality, and avoids decoding every slice just to inspect metadata.
    path = f"{series_dir}/{files[len(files) // 2]}"
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    except Exception:
        return None
    row = {"StudyInstanceUID": study_id, "SeriesInstanceUID": series_id, "n_slices": len(files)}
    for tag in HEADER_TAGS:
        value = getattr(ds, tag, None)
        if isinstance(value, (list, tuple)) or type(value).__name__ == "MultiValue":
            row[tag] = "|".join(str(v) for v in value)
        else:
            row[tag] = None if value is None else str(value)
    return row

def probe_study(study_id):
    study_dir = f"{ROOT}/train_series/{study_id}"
    rows = []
    for series_id in os.listdir(study_dir):
        row = read_one_series_header(study_id, series_id, study_dir)
        if row is not None:
            rows.append(row)
    return rows

with ThreadPoolExecutor(max_workers=16) as pool:
    header_rows = [row for rows in pool.map(probe_study, sample_ids) for row in rows]
headers = pd.DataFrame(header_rows)
print(f"read {len(headers):,} series headers from {headers['StudyInstanceUID'].nunique()} sampled studies")
headers[["n_slices", "SeriesDescription", "Manufacturer", "PixelSpacing", "Laterality"]].head(6)

In [ ]:
def populated_share(column):
    if column not in headers.columns:
        return 0.0
    series_col = headers[column]
    return 100.0 * (series_col.notna() & (series_col.astype(str).str.strip() != "")).mean()

tag_shares = sorted(((tag, populated_share(tag)) for tag in HEADER_TAGS), key=lambda t: t[1])
names = [t[0] for t in tag_shares]
values = [t[1] for t in tag_shares]
colors = [RED if v < 60 else ("#e0a53a" if v < 99.5 else BLUE) for v in values]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(len(names)), values, color=colors, edgecolor="white", linewidth=2, height=0.68)
for i, v in enumerate(values):
    ax.text(v + 1.3, i, f"{v:.1f}%", va="center", fontsize=9.5, color=RED if v < 60 else "#444")
ax.set_yticks(range(len(names)), names, fontsize=9.5)
ax.set_xlim(0, 118)
ax.set_title(f"Share of {len(headers):,} sampled series with each header tag populated", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

### Laterality

Five of the twelve labels are defined relative to the body's midline, so knowing which knee was
scanned is a prerequisite for using them correctly. `Laterality` is formally an optional DICOM
attribute, meaning scanners are allowed to leave it blank, and the measurement above shows how
often that happens in practice. The next cell checks two further things: whether laterality
availability differs by scanner manufacturer, since a gap that correlates with manufacturer is
a gap that also correlates with site and protocol rather than being random; and whether the tag
is ever present but blank, which a plain missing-value check would not catch.

In [ ]:
raw_lat = headers["Laterality"].fillna("").astype(str).str.strip().str.upper().str[:1]
headers["_lat"] = raw_lat.where(raw_lat.isin(["L", "R"]), "")

present_but_blank = int((headers["Laterality"].notna() &
                          (headers["Laterality"].astype(str).str.strip() == "")).sum())
by_study_has_lat = headers.groupby("StudyInstanceUID")["_lat"].apply(lambda s: (s != "").any())

print(f"studies with a usable L/R value on at least one series : "
      f"{by_study_has_lat.sum()}/{len(by_study_has_lat)} ({100 * by_study_has_lat.mean():.1f}%)")
print(f"series carrying a usable L/R value                     : {(headers['_lat'] != '').mean() * 100:.1f}%")
print(f"series where the tag is present but an empty string    : {present_but_blank}  (not the same as absent)")

by_vendor = (headers.assign(has_lat=headers["_lat"] != "")
             .groupby(headers["Manufacturer"].fillna("(missing)"))
             .agg(n_series=("has_lat", "size"), pct_with_lat=("has_lat", lambda s: 100 * s.mean()))
             .sort_values("n_series", ascending=False).head(8))

fig, ax = plt.subplots(figsize=(9, 3.4))
colors = [RED if v < 50 else BLUE for v in by_vendor["pct_with_lat"].values]
ax.barh(range(len(by_vendor))[::-1], by_vendor["pct_with_lat"].values, color=colors, edgecolor="white", linewidth=2, height=0.65)
for i, (v, n) in enumerate(zip(by_vendor["pct_with_lat"].values, by_vendor["n_series"].values)):
    ax.text(v + 1.5, len(by_vendor) - 1 - i, f"{v:.0f}% (n={n})", va="center", fontsize=9.5)
ax.set_yticks(range(len(by_vendor))[::-1], [v[:26] for v in by_vendor.index])
ax.set_xlim(0, 128)
ax.set_title("Laterality availability by scanner manufacturer", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

If availability clearly differs by manufacturer, the studies missing laterality are not
a random subset; they are systematically associated with whichever sites use the equipment that
tends to leave the tag blank. Where the tag is missing, laterality can often still be recovered
from geometry: DICOM patient coordinates follow the LPS convention, in which the positive x-axis
points toward the patient's left, so the sign of a slice's physical x-position indicates which
knee is present. Doing this correctly requires converting the stored corner position to the
centre of the slice first, and treating knees scanned very close to the midline as unresolved
rather than guessing, since the sign carries little information there.

### Slice ordering

The individual image files within a series are named by their SOP Instance UID, an identifier
assigned for uniqueness, not for order. Sorting a series' files alphabetically by filename
therefore produces an ordering with no necessary relationship to anatomical position. The
correct through-plane order can be recovered from each slice's position and orientation: given
the in-plane row and column direction vectors from `ImageOrientationPatient`, their cross
product gives the slice normal, and the dot product of that normal with `ImagePositionPatient`
gives a scalar that increases monotonically along the stack. `InstanceNumber` is a more direct
alternative where present, though it is not guaranteed to track physical position for
interleaved or multi-echo acquisitions, and it carries no sign, so it cannot on its own indicate
which end of a stack is which. The next cell compares filename order and `InstanceNumber` order
against this geometric ground truth on a sample of series.

In [ ]:
from scipy.stats import spearmanr

def physical_order_check(study_id, series_id, n_max=60):
    series_dir = f"{ROOT}/train_series/{study_id}/{series_id}"
    files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
    if len(files) < 6:
        return None
    through_plane_positions, instance_numbers = [], []
    for f in files:
        try:
            ds = pydicom.dcmread(f"{series_dir}/{f}", stop_before_pixels=True, force=True,
                                  specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
            orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)
            position = np.asarray(ds.ImagePositionPatient, dtype=float)
            normal = np.cross(orientation[:3], orientation[3:])
            through_plane_positions.append(float(np.dot(position, normal)))
            instance_numbers.append(float(ds.InstanceNumber))
        except Exception:
            return None
    return through_plane_positions, instance_numbers

rows = []
for _, r in headers.sample(min(60, len(headers)), random_state=0).iterrows():
    result = physical_order_check(r["StudyInstanceUID"], r["SeriesInstanceUID"])
    if result is None:
        continue
    positions, instance_numbers = result
    filename_rank = np.arange(len(positions))  # os.listdir + sorted() gives filename order
    rows.append({
        "filename_order":  abs(spearmanr(filename_rank, positions).statistic),
        "instance_number_order": abs(spearmanr(instance_numbers, positions).statistic),
    })
order_check = pd.DataFrame(rows, columns=["filename_order", "instance_number_order"]).dropna()

if len(order_check) == 0:
    print("No sampled series had at least six slices with readable geometry tags; "
          "increase the sample size or lower the six-slice threshold above to check this on "
          "the full dataset.")
else:
    fig, ax = plt.subplots(figsize=(9, 3.4))
    bins = np.linspace(0, 1, 26)
    ax.hist(order_check["filename_order"], bins=bins, alpha=0.85, color=RED,
            edgecolor="white", linewidth=0.5, label="sorted by filename")
    ax.hist(order_check["instance_number_order"], bins=bins, alpha=0.85, color=BLUE,
            edgecolor="white", linewidth=0.5, label="sorted by InstanceNumber")
    ax.legend(frameon=False, fontsize=9.5)
    ax.set_xlabel("|Spearman correlation| against true physical position")
    ax.set_ylabel("series")
    ax.set_title(f"Does each ordering track physical slice position? ({len(order_check)} series)", loc="left", fontweight="bold")
    plt.tight_layout(); plt.show()

    print(order_check.describe().loc[["mean", "50%", "min", "max"]].round(3).to_string())

If filename order shows essentially no relationship to physical position while
`InstanceNumber` order tracks it closely on most series, that has direct consequences for any
pipeline that stacks neighbouring slices as channels, or that samples "the middle slice" of a
series: both operations are only meaningful once the slices are in their true spatial order,
and the default result of listing files in a directory does not provide that.

### Pixel spacing and field of view

One more property worth checking before writing any preprocessing code: how much physical area
a single pixel covers, and how much anatomy the whole image covers, both of which vary with the
scanner and the acquisition protocol.

In [ ]:
pixel_spacing = pd.to_numeric(headers["PixelSpacing"].fillna("").str.split("|").str[0], errors="coerce")
n_rows = pd.to_numeric(headers["Rows"], errors="coerce")
field_of_view_mm = pixel_spacing * n_rows

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.4))

ax = axes[0]
ax.hist(pixel_spacing.dropna(), bins=50, color=BLUE, edgecolor="white", linewidth=0.5)
ax.set_xlabel("millimetres per pixel"); ax.set_ylabel("series")
ax.set_title(f"Pixel spacing (median {pixel_spacing.median():.3f} mm, "
             f"{pixel_spacing.max()/pixel_spacing.min():.1f}x spread)", loc="left", fontweight="bold")

ax = axes[1]
ax.hist(field_of_view_mm.dropna(), bins=50, color="#e1732c", edgecolor="white", linewidth=0.5)
ax.set_xlabel("millimetres covered"); ax.set_ylabel("series")
ax.set_title(f"Acquired field of view (median {field_of_view_mm.median():.0f} mm)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

If pixel spacing varies by several times across the sample, resizing every image to the
same pixel count without accounting for spacing hands a model anatomy at inconsistent physical
scale: the same number of pixels can represent very different amounts of real tissue from one
study to the next. This matters more for small findings, such as a meniscal tear on the order
of a few millimetres, than for large ones. Cropping to a fixed physical size in millimetres
using the pixel spacing, and only then resizing to a fixed pixel count, keeps the physical scale
consistent regardless of how the image was originally acquired. The baseline notebook applies
this idea directly.

## 7. A visual look at one study

Numbers only convey so much. This section renders every series belonging to a single study, at
the physically middle slice of each, sorted by the geometric ordering established above rather
than by filename, to make the plane and weighting differences concrete.

In [ ]:
def physically_sorted_files(study_id, series_id):
    series_dir = f"{ROOT}/train_series/{study_id}/{series_id}"
    files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
    keyed = []
    for f in files:
        try:
            ds = pydicom.dcmread(f"{series_dir}/{f}", stop_before_pixels=True, force=True,
                                  specific_tags=["ImagePositionPatient", "ImageOrientationPatient"])
            orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)
            position = np.asarray(ds.ImagePositionPatient, dtype=float)
            key = float(np.dot(position, np.cross(orientation[:3], orientation[3:])))
        except Exception:
            key = 0.0
        keyed.append((key, f))
    keyed.sort()
    return series_dir, [f for _, f in keyed]

def load_windowed_slice(series_dir, filename):
    ds = pydicom.dcmread(f"{series_dir}/{filename}", force=True)
    array = ds.pixel_array.astype(np.float32)
    low, high = np.percentile(array, [1, 99])
    image = np.clip((array - low) / max(high - low, 1e-6), 0, 1)
    description = str(getattr(ds, "SeriesDescription", "?"))[:28]
    spacing = float(ds.PixelSpacing[0]) if hasattr(ds, "PixelSpacing") else float("nan")
    return image, description, spacing

example_study = headers["StudyInstanceUID"].value_counts().index[0]
example_series = (train_series[train_series["StudyInstanceUID"] == example_study]
                   .sort_values(["Anatomical_Plane", "Fat_Suppression"]))

fig, axes = plt.subplots(1, len(example_series), figsize=(2.4 * len(example_series), 3.2))
axes = np.atleast_1d(axes)
for ax, (_, row) in zip(axes, example_series.iterrows()):
    ax.axis("off")
    series_dir, files = physically_sorted_files(example_study, row["SeriesInstanceUID"])
    if not files:
        continue
    image, description, spacing = load_windowed_slice(series_dir, files[len(files) // 2])
    ax.imshow(image, cmap="gray")
    ax.set_title(f"{row['Anatomical_Plane']}  fs={row['Fat_Suppression']}\n{description}",
                 fontsize=8.5, color=PLANE_COLORS[row["Anatomical_Plane"]])
    ax.text(0.5, -0.06, f"{len(files)} slices, {spacing:.2f} mm/px", transform=ax.transAxes,
            ha="center", va="top", fontsize=7.5, color="#666")
fig.suptitle(f"One study, all {len(example_series)} of its series", x=0.02, ha="left", fontsize=12.5, fontweight="bold", y=1.05)
plt.tight_layout(); plt.show()

The same knee looks markedly different depending on plane and weighting, which is the
concrete version of the earlier statistical observation that a study is a heterogeneous
collection of series rather than one image. This is also why models built for this task
generally define a fixed number of acquisition "slots," such as one input per plane, and fill
each from whichever series in the study best matches that slot, rather than expecting a fixed
number of raw series.

## 8. Summary and what it means for modelling

A few points from this exploration carry directly into how a model for this competition should
be built, and are picked up again in the baseline notebook.

* Direct image-level labels cover a small fraction of the training studies. The rest have to be
  turned into training targets by parsing the reports, so the quality of that parsing step
  matters as much as the choice of image model.
* The reports are multilingual. A weak-labelling approach restricted to one language will
  silently under-label the rest of the corpus rather than raising an error.
* Duplicate report text across studies means cross-validation folds should be grouped by report
  content, not assigned per study independently, to avoid a split that leaks identical targets
  across the train/validation boundary.
* A study's series are irregular in number, plane, and weighting. A model needs an explicit,
  fixed slot structure with a presence mask, rather than assuming every study looks the same
  shape.
* Laterality, needed for five of the twelve labels, is frequently missing from the DICOM header,
  and missing in a way that correlates with scanner manufacturer rather than at random. It can
  often be recovered geometrically from patient coordinates when the header tag is absent.
* Slice order within a series must be derived from geometry or `InstanceNumber`, not from
  filename, before any operation that depends on spatial adjacency or on selecting a
  representative slice.
* Pixel spacing varies enough across the dataset that a fixed pixel-count resize does not by
  itself produce a fixed physical scale; cropping to a fixed physical extent before resizing
  does.

None of this is a defect in the dataset so much as an accurate reflection of what a real,
multi-site clinical corpus looks like. The next notebook uses these observations to build a
first working baseline: a weak-label pipeline over the reports, a fixed-slot input built with
correct slice ordering and physical-scale cropping, a fold-safe validation split, and a simple
trainable model on top.